# 02 · The baseline, and the ceiling of prompting

**The decision this notebook supports:** should we fine-tune at all?

This is the most commercially important notebook in the lab and it contains no training. Fine-tuning is expensive, it is slow to iterate, and it creates an artefact somebody has to maintain. Before paying that, establish what the model does with no training and with better prompting.

Three conditions, same frozen evaluation:

1. **Stock, naive prompt.** The floor.
2. **Stock, engineered prompt.** Instructions about grounding and refusal, no documents.
3. **Stock, documents in context.** This is retrieval-augmented generation with perfect retrieval.

If condition 3 clears the bar, **retrieval is your answer and fine-tuning is the wrong tool.**

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

In [ ]:
# Load one unchanged base model and define the generator used by all conditions.
# transformers supplies the tokenizer and causal LM; torch supplies tensors and
# no-gradient execution. tqdm.auto provides a notebook-friendly progress bar.
# json formats later summaries; time is imported but not used by this generator.
# DEVICE is explicitly MPS here, so this original cell requires that backend.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, json
from tqdm.auto import tqdm

MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
# Use inference behavior (including dropout off) for the baseline. no_grad()
# inside generate separately prevents building the training gradient graph.
model.eval()

# The same function keeps decoding constant while we change only the system
# prompt. None means use the configured output cap; Python's 'or' also substitutes
# the default for other false-valued inputs, such as zero.
def generate(system, user, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CFG["inference"]["max_new_tokens"]
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    # Render the tokenizer's saved chat format with an assistant-start marker.
    # The following call tokenizes it; truncation=True can remove prompt tokens
    # beyond 3,072. This is a truncating cap, not an error-on-long-input check.
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
    # **ids expands input_ids and attention_mask into named generation arguments.
    # Greedy decoding adds at most max_new_tokens and can stop earlier on EOS.
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    # Select batch row zero, then slice away the prompt-length prefix. Decode only
    # the newly generated IDs and remove special control tokens and outer whitespace.
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("ready")

## The three system prompts

Condition 3 pastes every document into the context. At this corpus size that fits, which lets us measure the **ceiling** of retrieval rather than the performance of any particular retriever. Real retrieval is strictly worse than this, because it can fetch the wrong chunk.

In [ ]:
# Create three controlled input conditions: generic instructions, company-specific
# behavior without evidence, and the same behavior with all source documents.
# Parenthesized adjacent strings join automatically in Python. The generator
# inside join reads each sorted Markdown file and separates documents by blank lines.
# All-docs includes the available evidence, but more context can also distract a
# small model; it is not a guaranteed upper bound on a focused retriever.
NAIVE = "You are a helpful assistant."

ENGINEERED = (
    "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
    "Answer only from Meridian's official documents. If the documents do not contain the "
    "answer, say plainly that you do not have that information and offer to connect the "
    "customer to a dealer. Never guess a number. Never round a range figure upward."
)

DOCS = "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))
WITH_DOCS = ENGINEERED + "\n\n=== MERIDIAN OFFICIAL DOCUMENTS ===\n" + DOCS

# Count document tokens only. The actual chat input also contains the system
# instructions, separators, user question, and template control tokens.
print("document context:", len(tok(DOCS)['input_ids']), "tokens")

## Run the three conditions

Roughly 60 generations each on development questions. Expect several minutes per condition on a laptop. Start it and read the markdown below while it runs.

In [ ]:
# Measure a condition on the same development questions and the same scorer.
# fidx supplies reference facts; dev supplies questions and answerability labels.
# Keep both aggregate scores and raw answers so failures remain inspectable.
fidx = lab.fact_index()
dev = lab.eval_set("dev")

def run_condition(system, items, label):
    rows, raw = [], []
    # tqdm wraps the loop with progress reporting. Each iteration makes one local
    # model generation, then records its heuristic score and the verbatim answer.
    for q in tqdm(items, desc=label):
        a = generate(system, q["question"])
        rows.append(lab.score_answer(a, q, fidx))
        raw.append({"id": q["id"], "question": q["question"], "answer": a,
                    "answerable": q["answerable"], "expected": q["expected_value"]})
    # summarise keeps answerable accuracy and unanswerable refusal separate.
    # save_run persists the results under the supplied label; rerunning that label
    # overwrites its existing run file. The return tuple supports later comparison.
    s = lab.summarise(rows)
    lab.save_run(label, {"summary": s, "answers": raw})
    return s, raw


In [ ]:

# Execute the naive baseline now. The number of generations is len(dev)
# (currently 36), not the approximately 60 mentioned in the original prose.
s1, raw1 = run_condition(NAIVE,      dev, "01_stock_naive")
print(json.dumps(s1, indent=2))

In [ ]:
# Run the engineered-instruction baseline with no source documents.
# Only the system prompt changes from s1; question order, model, decoding, and
# scoring stay fixed. json.dumps pretty-prints the summary dictionary.
s2, raw2 = run_condition(ENGINEERED, dev, "02_stock_engineered")
print(json.dumps(s2, indent=2))

In [ ]:
# Run the all-documents condition. This supplies factual evidence in the prompt
# without training or changing any weights. Keep raw3 for question-level diagnosis;
# a summary alone cannot show product confusion or scorer mistakes.
s3, raw3 = run_condition(WITH_DOCS,  dev, "03_stock_with_docs")
print(json.dumps(s3, indent=2))

## Compare, then decide

In [ ]:
# pandas organizes the three condition summaries into a labelled DataFrame.
# Read the previously declared quality bar from disk, select comparable columns,
# and use the notebook-provided display function for a readable table.
# .loc selects a row by its label; this differs from selecting by integer position.
import pandas as pd
BAR = lab.load_runs()["00_bar"]["bar"]
df = pd.DataFrame([s1, s2, s3], index=["naive", "engineered", "with documents"])
cols = ["accuracy_answerable","hallucination_rate_answerable","over_refusal_rate","appropriate_refusal_rate"]
display(df[cols])

print("\nagainst the bar you set blind in notebook 01:")
best = df.loc["with documents"]
# Each comparison produces a boolean. The percent-format operator in strings
# (such as %.2f) formats the threshold; it is not a percentage calculation.
# all(checks.values()) is True only when every requirement passes.
checks = {
 "accuracy >= %.2f" % BAR["accuracy_answerable"]: best.accuracy_answerable >= BAR["accuracy_answerable"],
 "hallucination <= %.2f" % BAR["hallucination_rate_answerable"]: best.hallucination_rate_answerable <= BAR["hallucination_rate_answerable"],
 "appropriate refusal >= %.2f" % BAR["appropriate_refusal_rate"]: best.appropriate_refusal_rate >= BAR["appropriate_refusal_rate"],
 "over-refusal <= %.2f" % BAR["max_over_refusal"]: best.over_refusal_rate <= BAR["max_over_refusal"],
}
# The conditional expression 'PASS if v else FAIL' chooses the printed word.
# The verdict follows the heuristic metrics and declared bar; inspect failures
# before treating it as proof that a training technique is necessary.
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("\nVERDICT:", "prompting alone clears the bar — fine-tuning is NOT justified on quality grounds"
      if all(checks.values()) else "a gap remains that prompting did not close")

## Read the failures, not just the numbers

Aggregates tell you whether there is a problem. Only the individual failures tell you what kind of problem it is, and the kind determines the technique. Print ten.

In [ ]:
# Recover individual all-documents failures by pairing raw answers with dev
# questions using zip, rescoring each pair, and retaining scores not marked correct.
# The nested comprehensions depend on raw3 and dev remaining in the same order.
# [:10] limits displayed failures; [:150] shortens the displayed answer only.
# Scorer failures can include formatting/refusal-keyword issues, so inspect the
# actual answer and source before deciding what went wrong.
fails = [r for r, s in zip(raw3, [lab.score_answer(r["answer"], q, fidx)
         for r, q in zip(raw3, dev)]) if not s["correct"]]
for r in fails[:10]:
    print(f"Q: {r['question']}")
    print(f"   expected : {r['expected']}")
    print(f"   got      : {r['answer'][:150]}")
    print()

## The diagnosis that chooses the technique

Classify what you just read. Each failure mode has a different correct fix, and picking the wrong one wastes a week:

| What you see | What it means | The right fix |
|---|---|---|
| Right fact, wrong format or rambling | The knowledge is present, the behaviour is not | **SFT** |
| Cannot recall a fact even with documents in context | Retrieval or context handling, not knowledge | Better retrieval, or **CPT** if you need it without context |
| Confidently invents when it should decline | A preference problem, not a knowledge problem | **DPO** |
| Contradicts itself between answers | Data inconsistency | Fix the data first |

Write your own count below before moving on.

In [ ]:
# Record a human diagnosis after inspecting failures. None means not yet filled
# in; this cell does not automatically classify or count errors. Dictionary keys
# name the proposed categories, and save_run records your entries for later review.
# Executing unchanged saves the current None values, not a completed diagnosis.
DIAGNOSIS = {
    "format_or_verbosity": None,   # -> SFT
    "missing_knowledge":   None,   # -> CPT
    "invented_when_should_refuse": None,  # -> DPO
    "other": None,
}
# Fill these in from the failures above, then run this cell.
lab.save_run("04_diagnosis", {"diagnosis": DIAGNOSIS,
    "note": "Counted by hand from with-documents failures on the dev split."})
print(DIAGNOSIS)

## What you should be able to say now

- The measured cost of doing nothing, which is the only honest baseline.
- Whether prompting alone clears your bar, and what that implies about fine-tuning.
- Which failure mode dominates, and therefore which technique the next notebooks should emphasise.

**Next:** notebook 03 generates synthetic training data with Gemini and then curates it, because generated data that nobody checked is how quiet errors enter a model.